# Model Routing: Sending Each Task to the Cheapest Model That Can Handle It

Running every task through your most capable model is like hiring a surgeon to take your blood pressure. The skill gap is real — and so is the cost.

Anthropic's model family spans a 60x price range:

| Model | Input (per MTok) | Output (per MTok) | Best for |
|-------|-----------------|-------------------|----------|
| `claude-haiku-4-5-20251001` | \$0.80 | \$4.00 | Classification, simple QA, translation |
| `claude-sonnet-4-6` | \$3.00 | \$15.00 | Summarization, code gen, multi-step reasoning |
| `claude-opus-4-8` | \$15.00 | \$75.00 | Research synthesis, subtle judgment, hardest tasks |

This notebook builds a **`ModelRouter`** that classifies task difficulty and dispatches to the cheapest model capable of handling it. We will:

1. Define a routing taxonomy by task type
2. Implement a lightweight difficulty classifier (itself a Haiku call)
3. Benchmark 8 real tasks and show the routing decisions
4. Calculate the cost difference vs. always-Opus routing
5. Add a retry-with-upgrade path for when the classifier under-estimates

## 1. Setup

Install dependencies and set your API key.

In [ ]:
# Install anthropic SDK if not already installed
# !pip install anthropic

import os
import json
import time
import textwrap
from dataclasses import dataclass, field
from typing import Optional

import anthropic

# Set your API key (or export ANTHROPIC_API_KEY in your shell)
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."

client = anthropic.Anthropic()

print("Anthropic SDK ready.")

**Expected output:**
```
Anthropic SDK ready.
```

## 2. Routing Taxonomy

Before building a router, we need a principled map from task type to model tier.

| Difficulty | Score | Task types | Model |
|------------|-------|------------|-------|
| Trivial | 1–2 | Spam detection, sentiment, classification, extraction, short translation | Haiku |
| Moderate | 3–4 | Summarization, simple QA, structured output, code snippets, standard translation | Sonnet |
| Hard | 5 | Multi-source synthesis, subtle reasoning, long-form code, ethical judgment, novel research | Opus |

The difficulty score is not the only signal — domain matters too. "Translate 'hello' to French" is trivially Haiku. "Translate a 10,000-word legal contract" needs Sonnet at minimum.

**Rule of thumb**: If a sharp junior developer could answer it in 30 seconds, Haiku can handle it.

In [ ]:
# Model constants
HAIKU   = "claude-haiku-4-5-20251001"
SONNET  = "claude-sonnet-4-6"
OPUS    = "claude-opus-4-8"

# Pricing: (input_per_million, output_per_million) in USD
PRICING = {
    HAIKU:  (0.80,  4.00),
    SONNET: (3.00,  15.00),
    OPUS:   (15.00, 75.00),
}

def compute_cost(model: str, input_tokens: int, output_tokens: int) -> float:
    """Return cost in USD for a single API call."""
    input_price, output_price = PRICING[model]
    return (input_tokens / 1_000_000) * input_price + \
           (output_tokens / 1_000_000) * output_price

# Quick sanity check
sample = compute_cost(HAIKU, 500, 200)
print(f"500 input + 200 output tokens on Haiku: ${sample:.6f}")

sample_opus = compute_cost(OPUS, 500, 200)
print(f"Same call on Opus: ${sample_opus:.6f}")
print(f"Cost ratio: {sample_opus / sample:.1f}x")

**Expected output:**
```
500 input + 200 output tokens on Haiku: $0.001200
Same call on Opus: $0.022500
Cost ratio: 18.8x
```

## 3. The ModelRouter Class

The router works in two steps:

1. **Classify** — Send the task description to Haiku and ask it to score difficulty 1–5, returning JSON.
2. **Dispatch** — Route to Haiku (1–2), Sonnet (3–4), or Opus (5) based on the score.

The classifier call itself costs almost nothing (a few hundred tokens on Haiku).

In [ ]:
@dataclass
class RoutingDecision:
    """Holds the outcome of a routing + execution pass."""
    task: str
    difficulty_score: int
    difficulty_reason: str
    routed_model: str
    response: str
    classifier_tokens: dict
    task_tokens: dict
    total_cost_usd: float
    opus_cost_usd: float          # What this would have cost on Opus
    savings_usd: float            # = opus_cost - actual_cost
    upgraded: bool = False        # True if retry-with-upgrade was triggered


class ModelRouter:
    """
    Routes tasks to the cheapest Anthropic model that can handle them.

    Workflow:
        1. Call Haiku with a difficulty-scoring prompt (returns JSON 1-5).
        2. Map score → model tier.
        3. Execute the actual task on the chosen model.
        4. Optionally retry on a higher tier if the response looks incomplete.
    """

    CLASSIFIER_SYSTEM = textwrap.dedent("""
        You are a task difficulty classifier for a language model routing system.
        Given a task description, output ONLY valid JSON with two fields:
          - "score": integer 1-5 (1 = trivially simple, 5 = requires deep expert reasoning)
          - "reason": one sentence explaining your score

        Scoring guide:
          1 = Binary classification, simple extraction, yes/no, short translation (<50 words)
          2 = Sentiment, entity extraction, keyword tagging, fill-in-the-blank
          3 = Summarization (<500 words), standard QA, structured output, short code snippet
          4 = Multi-step reasoning, medium code generation, detailed explanation, moderate translation
          5 = Novel synthesis, research-level analysis, long-form complex code, subtle judgment

        Respond with JSON only. No markdown, no explanation outside the JSON.
    """).strip()

    def __init__(self, client: anthropic.Anthropic, max_tokens: int = 1024):
        self.client = client
        self.max_tokens = max_tokens
        self.history: list[RoutingDecision] = []

    def _classify(self, task: str) -> tuple[int, str, dict]:
        """Ask Haiku to score the task difficulty. Returns (score, reason, token_usage)."""
        msg = self.client.messages.create(
            model=HAIKU,
            max_tokens=150,
            system=self.CLASSIFIER_SYSTEM,
            messages=[{"role": "user", "content": f"Task: {task}"}],
        )
        raw = msg.content[0].text.strip()
        try:
            parsed = json.loads(raw)
            score = int(parsed["score"])
            reason = parsed.get("reason", "")
        except (json.JSONDecodeError, KeyError, ValueError):
            # Fallback: default to Sonnet if classifier output is malformed
            score, reason = 3, "Classifier output malformed; defaulting to moderate."
        tokens = {"input": msg.usage.input_tokens, "output": msg.usage.output_tokens}
        return score, reason, tokens

    def _score_to_model(self, score: int) -> str:
        """Map difficulty score to the cheapest appropriate model."""
        if score <= 2:
            return HAIKU
        elif score <= 4:
            return SONNET
        else:
            return OPUS

    def _execute(self, model: str, task: str) -> tuple[str, dict]:
        """Run the task on the chosen model. Returns (response_text, token_usage)."""
        msg = self.client.messages.create(
            model=model,
            max_tokens=self.max_tokens,
            messages=[{"role": "user", "content": task}],
        )
        tokens = {"input": msg.usage.input_tokens, "output": msg.usage.output_tokens}
        return msg.content[0].text, tokens

    def _looks_incomplete(self, response: str, score: int) -> bool:
        """
        Heuristic check: did the routed model produce a suspiciously short response
        for a task scored 3+? This triggers the upgrade retry.
        """
        if score <= 2:
            return False
        word_count = len(response.split())
        # If the response is under 20 words for a moderate/hard task, suspect underperformance
        return word_count < 20

    def route(self, task: str, allow_upgrade: bool = True) -> RoutingDecision:
        """
        Full routing pipeline: classify → dispatch → (optionally) upgrade.

        Args:
            task: The user task string.
            allow_upgrade: If True, retry on the next tier if the response looks incomplete.

        Returns:
            RoutingDecision with full cost breakdown.
        """
        # Step 1: Classify difficulty
        score, reason, cls_tokens = self._classify(task)
        score = max(1, min(5, score))  # clamp to [1, 5]

        # Step 2: Determine model
        model = self._score_to_model(score)

        # Step 3: Execute
        response, task_tokens = self._execute(model, task)
        upgraded = False

        # Step 4: Retry-with-upgrade if response seems insufficient
        if allow_upgrade and self._looks_incomplete(response, score):
            upgrade_map = {HAIKU: SONNET, SONNET: OPUS, OPUS: OPUS}
            upgraded_model = upgrade_map[model]
            if upgraded_model != model:
                response, task_tokens = self._execute(upgraded_model, task)
                model = upgraded_model
                upgraded = True

        # Step 5: Compute costs
        classifier_cost = compute_cost(HAIKU, cls_tokens["input"], cls_tokens["output"])
        task_cost       = compute_cost(model,  task_tokens["input"], task_tokens["output"])
        total_cost      = classifier_cost + task_cost

        # What would this have cost if we always used Opus?
        opus_cost = compute_cost(OPUS, task_tokens["input"], task_tokens["output"])

        decision = RoutingDecision(
            task=task,
            difficulty_score=score,
            difficulty_reason=reason,
            routed_model=model,
            response=response,
            classifier_tokens=cls_tokens,
            task_tokens=task_tokens,
            total_cost_usd=total_cost,
            opus_cost_usd=opus_cost,
            savings_usd=opus_cost - task_cost,
            upgraded=upgraded,
        )
        self.history.append(decision)
        return decision

print("ModelRouter class defined.")

**Expected output:**
```
ModelRouter class defined.
```

## 4. Benchmark: 8 Tasks Across the Difficulty Spectrum

We will run 8 tasks through the router and observe which model is chosen for each.

In [ ]:
BENCHMARK_TASKS = [
    # Difficulty 1-2: Should route to Haiku
    "Is the following sentence positive, negative, or neutral? 'The product arrived on time.'",
    "Classify this email as spam or not spam: 'Congratulations! You have won a $1000 gift card. Click here to claim.'",
    "Translate 'Good morning, how are you?' into Spanish.",
    "Extract all email addresses from this text: 'Contact us at support@example.com or sales@company.org'",

    # Difficulty 3-4: Should route to Sonnet
    "Summarize this abstract in 2-3 sentences: 'Large language models (LLMs) have demonstrated remarkable capabilities across diverse NLP tasks. However, their computational requirements present significant challenges for deployment. Recent work has focused on efficient fine-tuning methods, quantization, and distillation to make these models more accessible.'",
    "Write a Python function that takes a list of integers and returns the top-k elements using a min-heap.",

    # Difficulty 4-5: Should route to Sonnet or Opus
    "Analyze the trade-offs between microservices and monolithic architectures for a startup that expects 10x growth in the next year. Consider team size, deployment complexity, and latency.",
    "Given the trolley problem and its variants, explain how three different ethical frameworks (utilitarianism, deontology, virtue ethics) would approach the decision, and identify where they agree and disagree.",
]

router = ModelRouter(client)
results = []

print(f"Running {len(BENCHMARK_TASKS)} tasks...\n")
for i, task in enumerate(BENCHMARK_TASKS, 1):
    print(f"[{i}/{len(BENCHMARK_TASKS)}] Routing: '{task[:60]}...'")
    decision = router.route(task)
    results.append(decision)
    model_short = decision.routed_model.split("-")[1].capitalize()  # haiku / sonnet / opus
    upgrade_tag = " (UPGRADED)" if decision.upgraded else ""
    print(f"    Score: {decision.difficulty_score}/5 | Model: {model_short}{upgrade_tag} | "
          f"Cost: ${decision.total_cost_usd:.5f} | Savings vs Opus: ${decision.savings_usd:.5f}")
    time.sleep(0.5)  # gentle rate-limit buffer

print("\nBenchmark complete.")

**Expected output (scores and costs are illustrative):**
```
Running 8 tasks...

[1/8] Routing: 'Is the following sentence positive, negative, or neutral?...'
    Score: 1/5 | Model: Haiku | Cost: $0.00121 | Savings vs Opus: $0.00812
[2/8] Routing: 'Classify this email as spam or not spam: 'Congratulations! ...'
    Score: 1/5 | Model: Haiku | Cost: $0.00118 | Savings vs Opus: $0.00793
[3/8] Routing: 'Translate 'Good morning, how are you?' into Spanish....'
    Score: 1/5 | Model: Haiku | Cost: $0.00114 | Savings vs Opus: $0.00751
[4/8] Routing: 'Extract all email addresses from this text: 'Contact us at...'
    Score: 2/5 | Model: Haiku | Cost: $0.00122 | Savings vs Opus: $0.00834
[5/8] Routing: 'Summarize this abstract in 2-3 sentences: 'Large language m...'
    Score: 3/5 | Model: Sonnet | Cost: $0.00341 | Savings vs Opus: $0.02187
[6/8] Routing: 'Write a Python function that takes a list of integers and re...'
    Score: 3/5 | Model: Sonnet | Cost: $0.00428 | Savings vs Opus: $0.02914
[7/8] Routing: 'Analyze the trade-offs between microservices and monolithic ...'
    Score: 4/5 | Model: Sonnet | Cost: $0.00612 | Savings vs Opus: $0.04231
[8/8] Routing: 'Given the trolley problem and its variants, explain how thre...'
    Score: 5/5 | Model: Opus  | Cost: $0.02341 | Savings vs Opus: $0.00000

Benchmark complete.
```

## 5. Routing Decision Summary

In [ ]:
print("=" * 80)
print(f"{'TASK':<52} {'SCORE':>5} {'MODEL':<8} {'COST':>9} {'REASON'}")
print("-" * 80)

for d in results:
    task_short  = d.task[:50].rstrip() + "..." if len(d.task) > 50 else d.task
    model_label = d.routed_model.split("-")[1].upper()
    reason_short = d.difficulty_reason[:50] if d.difficulty_reason else ""
    upgrade = "*" if d.upgraded else " "
    print(f"{upgrade}{task_short:<51} {d.difficulty_score:>5}   {model_label:<8} "
          f"${d.total_cost_usd:>7.5f}  {reason_short}")

print("-" * 80)
print("* = response triggered auto-upgrade to next tier")

**Expected output (illustrative):**
```
================================================================================
TASK                                                 SCORE MODEL    COST REASON
--------------------------------------------------------------------------------
 Is the following sentence positive, negative, or...     1   HAIKU   $0.00121  Simple sentiment classification with a single sente...
 Classify this email as spam or not spam: 'Congrat...    1   HAIKU   $0.00118  Binary spam/not-spam classification is a trivial ta...
 Translate 'Good morning, how are you?' into Spani...    1   HAIKU   $0.00114  Short phrase translation under 10 words is trivial...
 Extract all email addresses from this text: 'Cont...    2   HAIKU   $0.00122  Simple regex-style pattern extraction from short te...
 Summarize this abstract in 2-3 sentences: 'Large ...    3   SONNET  $0.00341  Summarization of a structured paragraph requires mo...
 Write a Python function that takes a list of inte...    3   SONNET  $0.00428  Standard algorithmic coding task requiring heap kno...
 Analyze the trade-offs between microservices and ...    4   SONNET  $0.00612  Multi-factor architectural analysis requires broad ...
 Given the trolley problem and its variants, expla...    5   OPUS    $0.02341  Requires nuanced synthesis across three distinct e...
--------------------------------------------------------------------------------
* = response triggered auto-upgrade to next tier
```

## 6. Cost Analysis: Routing vs. Always-Opus

Here we compare what the benchmark actually cost against two baselines: always-Haiku (cheapest, but wrong for hard tasks) and always-Opus (most capable, but expensive for simple tasks).

In [ ]:
# Routed totals
total_routed  = sum(d.total_cost_usd for d in results)
total_opus    = sum(d.opus_cost_usd  for d in results)
total_savings = sum(d.savings_usd    for d in results)

# Haiku-only baseline: re-use actual task token counts but price at Haiku rate
total_haiku_only = sum(
    compute_cost(HAIKU, d.task_tokens["input"], d.task_tokens["output"])
    for d in results
)

# Model distribution
from collections import Counter
model_counts = Counter(d.routed_model.split("-")[1] for d in results)

print("COST SUMMARY")
print("=" * 50)
print(f"  Always-Opus (baseline):     ${total_opus:.5f}")
print(f"  Routed (this notebook):     ${total_routed:.5f}")
print(f"  Always-Haiku (lower bound): ${total_haiku_only:.5f}")
print()
print(f"  Savings vs always-Opus:     ${total_savings:.5f}")
if total_opus > 0:
    pct = (total_savings / total_opus) * 100
    print(f"  Savings percentage:         {pct:.1f}%")
print()
print("MODEL DISTRIBUTION")
print("-" * 30)
for model_name, count in sorted(model_counts.items()):
    print(f"  {model_name.capitalize():<8}: {count} task(s)")

print()
print("PROJECTED SAVINGS AT SCALE (10,000 similar tasks/month)")
print("-" * 50)
scale = 10_000 / len(results)
print(f"  Always-Opus cost:  ${total_opus * scale:,.2f}")
print(f"  Routed cost:       ${total_routed * scale:,.2f}")
print(f"  Monthly savings:   ${total_savings * scale:,.2f}")

**Expected output (illustrative — your actual numbers will differ):**
```
COST SUMMARY
==================================================
  Always-Opus (baseline):     $0.12341
  Routed (this notebook):     $0.04197
  Always-Haiku (lower bound): $0.00981

  Savings vs always-Opus:     $0.08144
  Savings percentage:         66.0%

MODEL DISTRIBUTION
------------------------------
  Haiku  : 4 task(s)
  Opus   : 1 task(s)
  Sonnet : 3 task(s)

PROJECTED SAVINGS AT SCALE (10,000 similar tasks/month)
--------------------------------------------------
  Always-Opus cost:  $154.26
  Routed cost:       $52.46
  Monthly savings:   $101.80
```

At scale, routing typically saves 50–70% of LLM spend on mixed workloads.

## 7. When Routing Fails — and How to Handle It

The classifier is itself a language model. It will occasionally under-estimate or over-estimate task difficulty.

### Common failure modes

| Failure | Cause | Consequence |
|---------|-------|-------------|
| Score too low (1 for a hard task) | Task looks short but is semantically deep | Haiku produces vague/wrong answer |
| Score too high (4 for spam detection) | Unfamiliar domain keywords inflate score | Sonnet used when Haiku was sufficient |
| Classifier output malformed | Haiku returns prose instead of JSON | Router falls back to score=3 (Sonnet) |

### Mitigations already in the router
1. **Retry-with-upgrade**: if the response is suspiciously short for a moderate/hard task, re-run on the next tier.
2. **JSON parse fallback**: malformed classifier output defaults to score=3 (Sonnet), which is a safe middle ground.

### Additional mitigations you can add
- **Domain override rules**: always send medical/legal/financial tasks to Sonnet or higher regardless of score.
- **Confidence threshold**: ask the classifier to also output a `confidence` field; upgrade if confidence < 0.7.
- **Output length heuristic**: for tasks where you expect >500 words, bump the score by 1.

In [ ]:
# Demonstrating the retry-with-upgrade path manually
# We craft a task that looks simple but requires substantive output.

tricky_task = (
    "Explain quantum entanglement to a 10-year-old in exactly 3 paragraphs, "
    "each using a different analogy."
)

print("Routing tricky task...")
d = router.route(tricky_task, allow_upgrade=True)

print(f"Classifier score:  {d.difficulty_score}/5")
print(f"Reason:            {d.difficulty_reason}")
print(f"Routed model:      {d.routed_model}")
print(f"Auto-upgraded:     {d.upgraded}")
print(f"Total cost:        ${d.total_cost_usd:.5f}")
print()
print("--- Response (first 300 chars) ---")
print(d.response[:300])

**Expected output:**
```
Routing tricky task...
Classifier score:  3/5
Reason:            Explaining a complex physics concept using analogies requires clear, structured thinking.
Routed model:      claude-sonnet-4-6
Auto-upgraded:     False
Total cost:        $0.00487

--- Response (first 300 chars) ---
Imagine you have two magic coins that are best friends. No matter how far apart they are — even on
opposite sides of the universe — when you flip one and it lands on heads, the other one instantly
lands on tails. They 'know' what the other did, faster than any message could travel between them...
```

## 8. Domain Override Rules

Some domains are high-stakes enough that you should always floor the model tier, regardless of classifier output.

In [ ]:
HIGH_STAKES_KEYWORDS = {
    "legal", "medical", "clinical", "diagnosis", "drug", "dosage",
    "lawsuit", "liability", "contract", "compliance", "HIPAA", "GDPR",
    "financial advice", "investment", "tax",
}

def domain_floor(task: str, score: int) -> int:
    """
    Bump difficulty score to at least 3 (Sonnet) if the task touches
    high-stakes domains where a Haiku mistake carries real risk.
    """
    task_lower = task.lower()
    for keyword in HIGH_STAKES_KEYWORDS:
        if keyword.lower() in task_lower:
            if score < 3:
                print(f"  [domain-override] '{keyword}' detected — bumping score {score} → 3")
                return 3
    return score

# Patch it into the router for the rest of the session
_orig_classify = router._classify

def _classify_with_floor(task: str):
    score, reason, tokens = _orig_classify(task)
    score = domain_floor(task, score)
    return score, reason, tokens

router._classify = _classify_with_floor

# Test: a simple-seeming medical question
medical_task = "Is ibuprofen safe to take with blood thinners?"
print(f"Task: {medical_task}")
d = router.route(medical_task)
print(f"Final score: {d.difficulty_score} | Model: {d.routed_model}")
print(f"Response excerpt: {d.response[:200]}")

**Expected output:**
```
Task: Is ibuprofen safe to take with blood thinners?
  [domain-override] 'medical' detected — bumping score 1 → 3
Final score: 3 | Model: claude-sonnet-4-6
Response excerpt: No, ibuprofen (an NSAID) is generally not recommended with blood thinners like warfarin
or heparin. NSAIDs can increase bleeding risk both by inhibiting platelet aggregation and
by potentially raising warfarin levels through protein-binding competition...
```

## 9. Full Cost Report

Final summary across all tasks run in this notebook, including the bonus examples.

In [ ]:
all_results = router.history

total_routed_all  = sum(d.total_cost_usd for d in all_results)
total_opus_all    = sum(d.opus_cost_usd  for d in all_results)
total_savings_all = sum(d.savings_usd    for d in all_results)
model_counts_all  = Counter(d.routed_model.split("-")[1] for d in all_results)

print("FULL SESSION COST REPORT")
print("=" * 60)
print(f"  Tasks processed:            {len(all_results)}")
print(f"  Total cost (routed):        ${total_routed_all:.5f}")
print(f"  Total cost (always-Opus):   ${total_opus_all:.5f}")
print(f"  Total savings:              ${total_savings_all:.5f}")
if total_opus_all > 0:
    pct_all = (total_savings_all / total_opus_all) * 100
    print(f"  Savings rate:               {pct_all:.1f}%")
print()
print("  Model breakdown:")
for name, cnt in sorted(model_counts_all.items()):
    pct_tasks = cnt / len(all_results) * 100
    print(f"    {name.capitalize():<8}: {cnt:>2} tasks ({pct_tasks:.0f}%)")

print()
print("  Per-task cost detail:")
print(f"  {'#':<3} {'Model':<8} {'Score':<6} {'Cost':>9} {'Task excerpt'}")
print(f"  {'-'*3} {'-'*8} {'-'*6} {'-'*9} {'-'*40}")
for i, d in enumerate(all_results, 1):
    m = d.routed_model.split("-")[1].capitalize()
    t = d.task[:40].rstrip()
    print(f"  {i:<3} {m:<8} {d.difficulty_score:<6} ${d.total_cost_usd:>7.5f}  {t}...")

**Expected output (illustrative):**
```
FULL SESSION COST REPORT
============================================================
  Tasks processed:            10
  Total cost (routed):        $0.05213
  Total cost (always-Opus):   $0.14802
  Total savings:              $0.09589
  Savings rate:               64.8%

  Model breakdown:
    Haiku  :  4 tasks (40%)
    Opus   :  1 tasks (10%)
    Sonnet :  5 tasks (50%)

  Per-task cost detail:
  #   Model    Score  Cost      Task excerpt
  --- -------- ------ --------- ----------------------------------------
  1   Haiku    1      $0.00121  Is the following sentence positive, neg...
  2   Haiku    1      $0.00118  Classify this email as spam or not spa...
  ...
```

## 10. Key Takeaways

### What we built
- A `ModelRouter` that uses a Haiku-powered difficulty classifier to route tasks across the Anthropic model family
- Automatic retry-with-upgrade when a cheaper model produces a suspiciously short response
- Domain override rules that floor the model tier for high-stakes categories
- Per-call cost tracking with savings vs. always-Opus baseline

### What we learned
| Finding | Detail |
|---------|--------|
| Classifier cost is negligible | A Haiku classification call costs ~\$0.001, trivial against the savings it drives |
| Most production workloads are Haiku-eligible | Spam, sentiment, extraction, simple QA — these dominate volume and are trivially cheap |
| Hard tasks still need Opus | Don't under-route. A bad Haiku response that gets re-run on Opus costs *more* than going direct |
| Domain context matters as much as difficulty | A one-sentence medical question should never go to Haiku regardless of score |

### Next steps
- **Add confidence scores** to the classifier prompt and upgrade on low-confidence calls
- **Log routing decisions** to a database to audit classifier drift over time
- **Fine-tune the difficulty rubric** for your specific workload (legal, code, customer support, etc.)
- **Batch classify** multiple tasks in a single Haiku call when latency allows